# URL Caller: a local tour

Open a page at a **deadline** (the requested time), or visit it early and **refresh** it at that time. The browser remains open for a person to continue.

This notebook uses only local URLs. Production calls can use any HTTP(S) destination supported by the browser; the mock server is optional. Expect about one minute for the small measurements below. They illustrate the tools, not a reliable p99 estimate.

**Before opening this notebook**, from the extracted project directory:

```sh
uv sync --locked --group notebook-only
uv run --locked --group notebook-only playwright install chromium
uv run --locked --group notebook-only jupyter lab notebooks/quickstart.ipynb
```

Run cells in order. Browser launch restrictions are reported explicitly. Interactive desktop opening is opt-in; the final graph states whether it measures a real browser or Python signaling only.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import subprocess
import sys
import tempfile

from IPython.display import Image, display
from timed_url_caller import Schedule, TimedBrowserCaller
from url_caller_demo import DemoServer

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').is_file() and (p / 'scripts').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the extracted project directory.')
RUNS = ROOT / '.notebook-runs'
RUNS.mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix='tour-', dir=RUNS))
print('Python:', sys.version.split()[0])
print('Package:', importlib.metadata.version('timed-url-caller'))
print('Outputs:', RUN)

# Use this kernel's Python. Subprocesses avoid Jupyter's active event-loop
# conflict with synchronous Playwright and support the progress process.
def run_cli(arguments):
    completed = subprocess.run([sys.executable, *arguments], cwd=ROOT,
                               capture_output=True, text=True)
    print(completed.stdout[-3000:])
    if completed.returncode:
        print(completed.stderr[-1500:])
    return completed


## 1. Select a mode and time

- **navigate:** prepare a ready tab; first visit to the destination happens at the deadline.
- **refresh:** visit now; reload the prepared document at the deadline. This needs connected Chromium, and an early visit can have effects on a real site.

`Schedule.after(15)` means 15 seconds from configuration. `Schedule.at(...)` takes an ISO timestamp with `Z` (UTC) or an offset such as `+01:00`. A **monotonic clock**, which does not jump with clock corrections, drives the final wait.

The default is **best-effort**: request once, even if the process wakes late. Set `max_late_ms=20` only if a late action should be cancelled. This is not a guarantee that a page loads within 20 ms.

In [ ]:
MODE = 'navigate'                  # Change to 'refresh' to try the second mode.
RUN_INTERACTIVE_DEMO = False       # Set True on your graphical desktop.
BROWSER_URL = 'http://127.0.0.1:9222'

# These objects only configure a call; construction makes no target request.
example = TimedBrowserCaller('http://127.0.0.1:8765/', Schedule.after(15),
                            mode=MODE,
                            browser_url=BROWSER_URL if MODE == 'refresh' else None,
                            progress=False)
print('Configured mode:', example.mode)
print('Best-effort:', example.schedule.max_late_ms is None)
# A script calls example.run() while its deadline is still in the future.
# With progress=True, put that call under if __name__ == '__main__':.


## 2. Use the independent mock server

The server binds to **loopback** (`127.0.0.1`, this computer). Port `0` asks the OS for a free port. It has the wisdom button and a live **CET** clock, meaning fixed UTC+1 here, without summer time.

The next cell starts the server, checks its page, optionally runs the timed caller, and always stops the server. For refresh, first run this command in a terminal and sign in if appropriate:

```sh
uv run --no-sync scripts/open_browser.py
```

The launcher enables **CDP** (Chrome DevTools Protocol), a local connection used to control Chromium. Its address is separate from the destination. The progress bar and primed spinner run in another process; a short schedule may show the spinner immediately.

For a longer hands-on demo, keep `uv run --no-sync scripts/mock_server.py` running in a terminal instead.

The notebook captures command output. Run the CLI in a terminal to see animated progress live. `--no-sync` keeps this already prepared notebook environment unchanged.

In [ ]:
import threading
from urllib.request import ProxyHandler, build_opener

with DemoServer(port=0) as server:
    worker = threading.Thread(target=server.serve_forever, daemon=True)
    worker.start()
    try:
        print('Mock URL:', server.url)
        with build_opener(ProxyHandler({})).open(server.url, timeout=2) as response:
            page = response.read().decode()
        assert 'Click me for some wisdom.' in page
        print('Local HTTP page: OK')
        if RUN_INTERACTIVE_DEMO:
            arguments = ['-m', 'timed_url_caller', server.url, '--after', '15', '--mode', MODE]
            if MODE == 'refresh':
                arguments += ['--browser-url', BROWSER_URL]
            result = run_cli(arguments)
            if result.returncode:
                raise RuntimeError('Interactive demo failed; read the message above.')
            print('The browser tab stays open. Its wisdom button runs locally in the page.')
        else:
            print('Interactive opening skipped. Set RUN_INTERACTIVE_DEMO=True to enable it.')
    finally:
        server.shutdown()
        worker.join()


## 3. Run the tests

Fast tests cover input validation, timing policy, one-shot behavior, real local HTTP/event delivery, fake-controller contracts and benchmark exports. Real Chromium tests are opt-in and show as skipped here; this is not browser validation. To run them from a terminal, set `RUN_BROWSER_TESTS=1` and use the browser dependency group (see docs/testing.md for Windows syntax).

In [ ]:
tests = run_cli(['-m', 'unittest', 'discover', '-s', 'tests', '-v'])
print(tests.stderr[-700:])  # unittest writes its summary to stderr.
assert tests.returncode == 0, 'Fix failed tests before relying on measurements.'


## 4. Compare real browser modes locally

The harness owns a private Chromium instance and its own mock server. It compares navigate via **SSE** (a server-sent event to a ready tab), navigate via CDP, and refresh via CDP, with progress on/off. It checks that refresh is a real reload and that the button and clock work.

The three timings are **Python dispatch**, **document arrival** at the server, and **interactive receipt** (a per-document browser beacon after two animation frames). The last is a readiness proxy, not physical screen paint. Failed and slow samples are retained.

Three samples per case keep the tour short; use at least 30 for an initial comparison and more for tail estimates. A missing or blocked Chromium produces a failure report and no browser graph.

In [ ]:
BROWSER_EXECUTABLE = None  # Optional full path to installed Chromium.
browser_output = RUN / 'browser'
arguments = ['-m', 'url_caller_bench', '--samples', '3', '--lead-seconds', '1',
             '--output', str(browser_output)]
if BROWSER_EXECUTABLE:
    arguments += ['--executable', BROWSER_EXECUTABLE]
browser_run = run_cli(arguments)
browser_state = json.loads((browser_output / 'run_status.json').read_text())
print('Browser benchmark:', browser_state['status'])
if browser_state['failure']:
    print('Blocked/failed stage:', browser_state['failure']['stage'])
    print(browser_state['failure']['message'][:800])


## 5. Isolate Python signaling overhead

This is a separate diagnostic, useful even when Chromium cannot launch. A Python process receives the real local event, but **never opens the destination**. It measures no navigation, reload or rendering. Keep its graph separate from browser results.

All delay axes and numeric tick labels use **milliseconds (ms)**. Raw `_ns` fields use **nanoseconds (ns)**. The horizontal axis identifies cases and is unitless.

The plot marks individual samples, median, p95 (95% of successful values are at or below it) and a 20 ms reference line. Read failure counts as well as percentiles.

In [ ]:
signal_output = RUN / 'signal'
signal_run = run_cli(['-m', 'url_caller_bench', '--signal-only', '--samples', '5',
                      '--lead-seconds', '0.5', '--output', str(signal_output)])
signal_state = json.loads((signal_output / 'run_status.json').read_text())
print('Python-only diagnostic:', signal_state['status'])


## 6. Read the graph and continue

The next cell selects the real-browser graph if one was produced; otherwise it labels and shows the Python-only diagnostic. PNG/SVG, raw CSV and JSON summaries are saved under the output directory printed above. No failed browser run is relabeled as successful browser data.

For real use, replace the local URL with your destination, choose a future time and keep the tab visible and unchanged. The browser and network can add latency outside Python's control. See `docs/index.md` for searchable setup, API, timing and troubleshooting details.

In [ ]:
if browser_state['performance_plot_generated']:
    chosen = browser_output
    print('REAL BROWSER measurement:', browser_state['status'])
    print('Check failure counts and the summary before drawing conclusions.')
elif signal_state['performance_plot_generated']:
    chosen = signal_output
    print('PYTHON SIGNALING ONLY — no browser timing was measured.')
else:
    chosen = None
    print('No successful measurements; inspect the saved failure reports.')
if chosen:
    display(Image(filename=str(chosen / 'performance.png')))
    print('Detailed results:', chosen / 'summary.json')
